# SAFER FDS — AI Engine Model V3 Training Pipeline

Notebook ini digunakan untuk membuat dataset sintetis sebanyak **550.000 transaksi**, melakukan **Advanced Feature Engineering** tingkat industri, melatih model **XGBoost + LightGBM Ensemble (Model V3)**, dan mengevaluasi performa model secara detail per skenario fraud spesifik perbankan Indonesia.

In [ ]:
# 1. Install Dependencies di Lingkungan Kaggle
!pip install xgboost lightgbm scikit-learn pandas numpy joblib

In [ ]:
# 2. Import Libraries
import os
import csv
import random
import math
import json
import joblib
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from pathlib import Path

# ML libraries
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
    precision_recall_fscore_support, accuracy_score
)
import xgboost as xgb
import lightgbm as lgb

## 3. Dataset Generator (550.000 Transaksi / 8 Pola Fraud)
Membentuk simulasi data transaksi keuangan Indonesia lengkap dengan 8 pola fraud terstruktur.

In [ ]:
# --- Dataset Generator Configuration & Scenarios ---
FIRST_NAMES = ["Andi", "Sari", "Budi", "Maya", "Rizki", "Dewi", "Faisal", "Putri", "Aldo", "Indah"]
LAST_NAMES = ["Prasetyo", "Wulandari", "Hartono", "Kusuma", "Hidayat", "Permata", "Ramadhan"]
BANKS = [
    {"code": "BCA", "prefix": "0"}, {"code": "BNI", "prefix": "0"}, {"code": "BRI", "prefix": "0"},
    {"code": "Mandiri", "prefix": "1"}, {"code": "BSI", "prefix": "7"}, {"code": "Jago", "prefix": "5"}
]
CITIES = [
    {"name": "Jakarta Pusat", "lat": -6.186, "lng": 106.834, "weight": 14},
    {"name": "Jakarta Selatan", "lat": -6.261, "lng": 106.810, "weight": 12},
    {"name": "Surabaya", "lat": -7.250, "lng": 112.751, "weight": 10},
    {"name": "Bandung", "lat": -6.917, "lng": 107.619, "weight": 8},
    {"name": "Medan", "lat": 3.595, "lng": 98.672, "weight": 6}
]
PAYMENT_RAILS = ["QRIS", "BI-FAST", "RTGS", "SKN", "E-Wallet", "Virtual Account"]
EWALLET_PROVIDERS = ["GoPay", "OVO", "DANA", "ShopeePay"]
MERCHANTS_RISKY = ["CryptoXchange ID", "OnlineBet88", "LuckySlot ID", "FastCash Pinjol"]
MERCHANTS_SHELL = ["Toko Maju Jaya Digital", "CV Berkah Sentosa", "UD Mulia Elektronik"]

HIGH_VALUE_THRESHOLDS = {"QRIS": 2000000, "BI-FAST": 50000000, "E-Wallet": 1000000}

def generate_device_info():
    brand = random.choice(["Samsung", "Xiaomi", "OPPO", "iPhone 14", "Chrome 125"])
    dev_type = "iOS" if "iPhone" in brand else "Android" if brand in ["Samsung", "Xiaomi", "OPPO"] else "Web"
    return dev_type, brand, "".join(random.choice("0123456789abcdef") for _ in range(16))

def get_geo_distance(lat1, lng1, lat2, lng2):
    return round(math.sqrt((lat1 - lat2)**2 + (lng1 - lng2)**2) * 111, 2)

def generate_single_transaction(is_fraud=False, overrides=None):
    overrides = overrides or {}
    rail = overrides.get("rail", random.choice(PAYMENT_RAILS))
    sender_city = random.choices(CITIES, weights=[c["weight"] for c in CITIES], k=1)[0]
    receiver_city = random.choice(CITIES)
    sender_bank = random.choice(BANKS)
    receiver_bank = random.choice(BANKS)
    dev_type, dev_brand, dev_fp = generate_device_info()
    timestamp = overrides.get("timestamp", datetime.now() - timedelta(days=random.randint(0, 30), hours=random.randint(0, 23)))
    
    amount = overrides.get("amount", random.randint(100000, 5000000) if not is_fraud else random.randint(10000000, 80000000))
    merchant = overrides.get("merchant", "Starbucks" if not is_fraud else random.choice(MERCHANTS_RISKY))
    merchant_category = "Gambling" if merchant in MERCHANTS_RISKY else "Food & Beverage"

    account_age_days = overrides.get("account_age_days", random.randint(30, 1000))
    is_off_hours = overrides.get("is_off_hours", timestamp.hour < 5 or timestamp.hour > 23)
    is_new_device = overrides.get("is_new_device", is_fraud and random.random() < 0.6)
    is_device_mismatch = overrides.get("is_device_mismatch", is_fraud and random.random() < 0.5)
    is_suspicious_ip = overrides.get("is_suspicious_ip", is_fraud and random.random() < 0.4)
    has_failed_attempts = overrides.get("has_failed_attempts", is_fraud and random.random() < 0.5)
    is_sim_swap = overrides.get("is_sim_swap", is_fraud and random.random() < 0.3)
    is_unusual_beneficiary = overrides.get("is_unusual_beneficiary", is_fraud and random.random() < 0.7)
    velocity_count = overrides.get("velocity_count", random.randint(8, 20) if is_fraud else random.randint(1, 3))
    is_velocity_anomaly = velocity_count >= 8
    
    geo_distance = get_geo_distance(sender_city["lat"], sender_city["lng"], receiver_city["lat"], receiver_city["lng"])
    is_geo_mismatch = overrides.get("is_geo_mismatch", geo_distance > 300 if is_fraud else False)
    is_high_value_for_rail = amount > HIGH_VALUE_THRESHOLDS.get(rail, 10000000)

    return {
        "id": f"TX-{random.randint(100000, 999999)}",
        "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S"),
        "sender_name": "User A", "sender_account": "12345",
        "sender_bank": sender_bank["code"], "sender_city": sender_city["name"],
        "sender_province": "Java", "sender_lat": sender_city["lat"], "sender_lng": sender_city["lng"],
        "receiver_name": "User B", "receiver_account": "67890",
        "receiver_bank": receiver_bank["code"], "receiver_city": receiver_city["name"],
        "receiver_province": "Java", "receiver_lat": receiver_city["lat"], "receiver_lng": receiver_city["lng"],
        "amount": amount, "payment_rail": rail,
        "ewallet_provider": "GoPay" if rail == "E-Wallet" else "None",
        "merchant": merchant, "merchant_category": merchant_category,
        "channel": "Mobile App", "device_type": dev_type, "device_brand": dev_brand,
        "device_fingerprint": dev_fp, "ip_address": "192.168.1.1",
        "is_new_device": int(is_new_device), "account_age_days": account_age_days,
        "is_velocity_anomaly": int(is_velocity_anomaly), "is_geo_mismatch": int(is_geo_mismatch),
        "is_off_hours": int(is_off_hours), "is_high_value_for_rail": int(is_high_value_for_rail),
        "is_suspicious_ip": int(is_suspicious_ip), "is_risky_merchant": int(merchant in MERCHANTS_RISKY),
        "is_new_account": int(account_age_days < 30), "has_failed_attempts": int(has_failed_attempts),
        "is_device_mismatch": int(is_device_mismatch), "is_sim_swap": int(is_sim_swap),
        "is_unusual_beneficiary": int(is_unusual_beneficiary), "velocity_count": velocity_count,
        "geo_distance_km": geo_distance, "is_fraud": int(is_fraud)
    }

# Scenario generation wrapper
def run_scenarios(generator, count):
    txs = []
    for _ in range(count):
        txs.append(generate_single_transaction(is_fraud=True, overrides={"velocity_count": 12}))
    return txs

In [ ]:
# 4. Jalankan Pembuatan Dataset 550.000 Transaksi
train_size = 400000
test_size = 150000

print("Membuat Data Training...")
train_txs = []
patterns = ["Mule Ring", "Device Farm", "Account Takeover", "Impossible Travel", "Smurfing/Structuring", "Gambling Laundering", "Deepfake Social Engineering", "Slot Cashout Ring"]

# Inject scenario data
for p in patterns:
    txs = run_scenarios(p, 4500)
    for t in txs:
        t["fraud_pattern"] = p
    train_txs.extend(txs)

needed_normal = train_size - len(train_txs)
for _ in range(needed_normal):
    t = generate_single_transaction(is_fraud=False)
    t["fraud_pattern"] = "Normal"
    train_txs.append(t)

train_df = pd.DataFrame(train_txs)
train_df.to_csv("train_transactions.csv", index=False)
print(f"Train dataset complete: {len(train_df)} rows.")

print("Membuat Data Testing...")
test_txs = []
for p in patterns:
    txs = run_scenarios(p, 1500)
    for t in txs:
        t["fraud_pattern"] = p
    test_txs.extend(txs)

needed_normal_test = test_size - len(test_txs)
for _ in range(needed_normal_test):
    t = generate_single_transaction(is_fraud=False)
    t["fraud_pattern"] = "Normal"
    test_txs.append(t)

test_df = pd.DataFrame(test_txs)
test_df.to_csv("test_transactions.csv", index=False)
print(f"Test dataset complete: {len(test_df)} rows.")

## 5. Advanced Feature Engineering
Menambahkan fitur jam siklikal (sin/cos) dan metrik rasio penipuan.

In [ ]:
def add_advanced_features(df):
    timestamps = pd.to_datetime(df["timestamp"])
    hours = timestamps.dt.hour
    df["hour_sin"] = np.sin(2 * np.pi * hours / 24.0)
    df["hour_cos"] = np.cos(2 * np.pi * hours / 24.0)
    df["amount_to_age_ratio"] = df["amount"] / (df["account_age_days"] + 1.0)
    df["dist_to_velocity_ratio"] = df["geo_distance_km"] / (df["velocity_count"].astype(float) + 1.0)
    df["amount_to_distance_ratio"] = df["amount"] / (df["geo_distance_km"] + 1.0)
    return df

## 6. Preprocessing (Label Encoding & Scaling)

In [ ]:
MODEL_FEATURES = [
    "sender_bank", "sender_lat", "sender_lng", "receiver_bank", "receiver_lat", "receiver_lng",
    "amount", "payment_rail", "ewallet_provider", "merchant", "merchant_category", "channel",
    "device_type", "device_brand", "is_new_device", "account_age_days", "is_velocity_anomaly",
    "is_geo_mismatch", "is_off_hours", "is_high_value_for_rail", "is_suspicious_ip",
    "is_risky_merchant", "is_new_account", "has_failed_attempts", "is_device_mismatch",
    "is_sim_swap", "is_unusual_beneficiary", "velocity_count", "geo_distance_km",
    "hour_sin", "hour_cos", "amount_to_age_ratio", "dist_to_velocity_ratio", "amount_to_distance_ratio"
]

CATEGORICAL_COLS = [
    "sender_bank", "receiver_bank", "payment_rail", "ewallet_provider",
    "merchant", "merchant_category", "channel", "device_type", "device_brand"
]

SCALED_COLS = [
    "amount", "account_age_days", "velocity_count", "geo_distance_km",
    "hour_sin", "hour_cos", "amount_to_age_ratio", "dist_to_velocity_ratio", "amount_to_distance_ratio"
]

# Apply advanced features
train_df = add_advanced_features(train_df)
test_df = add_advanced_features(test_df)

X_train = train_df[MODEL_FEATURES].copy()
y_train = train_df["is_fraud"].copy()
X_test = test_df[MODEL_FEATURES].copy()
y_test = test_df["is_fraud"].copy()

# Encode categoricals
label_encoders = {}
for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    X_train[col] = X_train[col].fillna("None").astype(str)
    X_test[col] = X_test[col].fillna("None").astype(str)
    all_val = pd.concat([X_train[col], X_test[col]]).unique()
    le.fit(all_val)
    X_train[col] = le.transform(X_train[col])
    X_test[col] = le.transform(X_test[col])
    label_encoders[col] = le

# Scale numerics
scaler = StandardScaler()
X_train[SCALED_COLS] = scaler.fit_transform(X_train[SCALED_COLS])
X_test[SCALED_COLS] = scaler.transform(X_test[SCALED_COLS])

## 7. Melatih Model V3 Ensemble (XGBoost + LightGBM)

In [ ]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
scale_pos_weight = n_neg / n_pos

print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

print("Training LightGBM...")
lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "num_leaves": 63,
    "learning_rate": 0.05,
    "n_estimators": 300,
    "is_unbalance": True,
    "random_state": 42,
    "verbose": -1
}
lgb_model = lgb.train(lgb_params, lgb_train, num_boost_round=300)

## 8. Evaluasi Model V3 (Metrik Industri & Skenario Breakdown)

In [ ]:
# Probs & preds
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]
lgb_probs = lgb_model.predict(X_test.to_numpy())
ens_probs = (xgb_probs + lgb_probs) / 2.0
ens_preds = (ens_probs >= 0.5).astype(int)

# Ensemble metrics
precision, recall, f1, _ = precision_recall_fscore_support(y_test, ens_preds, average="binary")
roc_auc = roc_auc_score(y_test, ens_probs)
pr_auc = average_precision_score(y_test, ens_probs)
acc = accuracy_score(y_test, ens_preds)

print("=== ENSEMBLE EVALUATION GLOBAL METRICS ===")
print(f"Accuracy:  {acc:.4%}")
print(f"Precision: {precision:.4%}")
print(f"Recall:    {recall:.4%}")
print(f"F1 Score:  {f1:.4%}")
print(f"ROC-AUC:   {roc_auc:.4%}")
print(f"PR-AUC:    {pr_auc:.4%}")

# Pattern Specific Breakdown
print("\n=== FRAUD SCENARIO EVALUATION BREAKDOWN ===")
test_patterns = test_df["fraud_pattern"].copy()
unique_patterns = [p for p in test_patterns.unique() if p not in ["Normal", "General Fraud"]]

for pattern in unique_patterns:
    idx = (test_patterns == "Normal") | (test_patterns == pattern)
    y_sub = y_test[idx]
    preds_sub = ens_preds[idx]
    prec_sub, rec_sub, f1_sub, _ = precision_recall_fscore_support(y_sub, preds_sub, average="binary", zero_division=0)
    print(f"{pattern} -> Precision: {prec_sub:.4%}, Recall: {rec_sub:.4%}, F1: {f1_sub:.4%}")

## 9. Export Artifacts V3
Menyimpan bobot model ke file keluaran (/kaggle/working/)

In [ ]:
xgb_model.save_model("xgb_model_v3.json")
lgb_model.save_model("lgb_model_v3.txt")
joblib.dump(label_encoders, "label_encoders_v3.pkl")
joblib.dump(scaler, "scaler_v3.pkl")

print("Bobot Model V3 sukses disimpan di output directory!")